In [ ]:
# Import core libraries for data handling, visualization, and modeling
import pandas as pd
import shap
import matplotlib.pyplot as plt
import numpy as np

# Import sklearn metrics, preprocessing tools, model, and model selection utilities
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV

# Import Keras (TensorFlow) components for building a neural network
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Recall, Precision, F1Score, AUC
from tensorflow.keras.regularizers import l1

In [ ]:
# Load the COVID dataset from CSV into a pandas DataFrame
df = pd.read_csv('covid.csv')

In [ ]:
# Inspect the structure of the DataFrame: column names, types, and non‑null counts
df.info()

In [ ]:
# Preview the first rows to understand the contents and value patterns
df.head()

In [ ]:
# 1) Collect all columns whose name contains 'Date'
cols_to_drop = df.filter(regex='Date').columns.to_list()

# 2) Count missing values per column and sort descending
s = df.isna().sum().sort_values(ascending=False)

# 3) Add all columns that have at least one missing value to the drop list
cols_to_drop = list(set(cols_to_drop + list(s[s > 0].index)))

In [ ]:
# Manually add ID and Region columns to the drop list (not used as features)
cols_to_drop.append('Patient_ID')
cols_to_drop.append('Region')

In [ ]:
# Drop all selected columns from the DataFrame in place
df.drop(
    columns=cols_to_drop,
    inplace=True
)

In [ ]:
# Select names of all columns that are of 'object' dtype (categorical/text)
obj_cols = df.select_dtypes(include='object').columns
# Convert all object columns to pandas 'string' dtype for consistent handling
df[obj_cols] = df[obj_cols].astype('string')

In [ ]:
# Confirm updated dtypes after the conversion to string
df.info()

In [ ]:
# Initialize scaler for numeric ranges [0,1] and an ordinal encoder for categories
scaler = MinMaxScaler(feature_range=(0, 1))
oe = OrdinalEncoder()

# Apply: first encode categorical columns, then scale all columns,
# and rebuild a DataFrame with the same column names
final = pd.DataFrame(
    data=scaler.fit_transform(oe.fit_transform(df)),
    columns=df.columns
)

In [ ]:
# Separate target variable (Recovered) from the feature set
target = final['Recovered']
final = final.drop(columns='Recovered')

In [ ]:
# Split data into train and test sets (70/30) with a fixed random seed
# Note: no stratification is used here
x_tr, x_te, y_tr, y_te = train_test_split(
    final,
    target,
    shuffle=True,
    random_state=42,
    test_size=0.3
)

In [ ]:
# Define hyperparameter grid for Logistic Regression (regularization strength & solver)
param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['saga', 'sag', 'lbfgs']
}

In [ ]:
# Define multiple scoring metrics to evaluate during GridSearchCV
scoring = {
    'accuracy': 'accuracy',
    'f1': 'f1',
    'recall': 'recall',
    'precision': 'precision'
}

In [ ]:
# Configure GridSearchCV to tune LogisticRegression using 5‑fold CV
# The best model will be chosen based on accuracy ('refit' metric)
search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    cv=5,
    param_grid=param_grid,
    refit='accuracy',
    scoring=scoring,
    n_jobs=-1
)

In [ ]:
# Run the grid search on the training data to find the best hyperparameters
search.fit(x_tr, y_tr)

In [ ]:
# Extract the best LogisticRegression model found by GridSearchCV
model = search.best_estimator_

In [ ]:
# Fit the best model on the full training set (in case refit did not already do it)
model.fit(x_tr, y_tr)

In [ ]:
# Get predicted class labels on the test set
y_pred = model.predict(x_te)

In [ ]:
# Print precision, recall, f1-score and support for each class
print(classification_report(y_te, y_pred))

In [ ]:
# Build a SHAP explainer for the linear model and compute SHAP values on the test set
explainer = shap.LinearExplainer(model, x_te)
shap_values = explainer(x_te)

In [ ]:
# Visual SHAP summary plot (beeswarm) to show feature impact distributions
shap.summary_plot(shap_values, x_te)

In [ ]:
# SHAP bar plot summarizing mean absolute importance of each feature
shap.summary_plot(shap_values, x_te, plot_type='bar')

In [ ]:
# Predict class probabilities for the positive class (1) on the test set
y_proba = model.predict_proba(x_te)[:, 1]

# Compute ROC curve coordinates and AUC value
fpr, tpr, threshold = roc_curve(y_te, y_proba)
roc_auc = auc(fpr, tpr)

# For each threshold, convert probabilities to labels and compute F1 score
f1_roc = []
for th in threshold:
    y_predict = (y_proba >= th).astype('int')
    tp = np.sum((y_predict == 1) & (y_te == 1))
    tn = np.sum((y_predict == 0) & (y_te == 0))
    fp = np.sum((y_predict == 1) & (y_te == 0))
    fn = np.sum((y_predict == 0) & (y_te == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    f1_roc.append(f1score)

# Find the ROC point (FPR, TPR) corresponding to the threshold with maximum F1
best_idx = np.argmax(f1_roc)
best_fpr = fpr[best_idx]
best_tpr = tpr[best_idx]

In [ ]:
# Plot the ROC curve with the best F1 point highlighted via horizontal/vertical lines
plt.figure(figsize=(15, 8))
plt.plot(fpr, tpr, lw=2, label=f'Roc curve auc: {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], color='lightgray', linestyle='--', label='Random Classifier')
plt.axhline(y=best_tpr, lw=2, color='green', label='True Positive Rate', linestyle='--')
plt.axvline(x=best_fpr, lw=2, color='red', label='False Positive Rate', linestyle='--')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.legend()
plt.grid(True, alpha=0.4)

Sequential Neural Network

In [ ]:
# Define a deep neural network model (Sequential) for binary classification
ml = Sequential(
    [
        Input(shape=(14,)),                                   # input layer with 14 features
        Dense(units=128, activation='relu', kernel_regularizer=l1(0.01)),
        Dropout(0.3),
        Dense(units=64, activation='relu', kernel_regularizer=l1(0.0001)),
        Dropout(0.2),
        Dense(units=32, activation='relu', kernel_regularizer=l1(0.001)),
        Dropout(0.4),
        Dense(units=16, activation='relu', kernel_regularizer=l1(0.0001)),
        Dropout(0.3),
        Dense(units=1, activation='sigmoid')                  # output layer
    ]
)


In [ ]:
# Compile the neural network with metrics and optimizer
ml.compile(
    metrics=[
        'accuracy',
        Recall(name='recall'),
        Precision(name='precision'),
        F1Score(name='f1-score'),
        AUC(name='auc')
    ],
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.001)
)

In [ ]:
# Print a summary of the neural network architecture
ml.summary()

In [ ]:
# Train the neural network on the training data with validation split
history = ml.fit(
    x_tr,
    y_tr,
    epochs=20,
    batch_size=15,
    verbose=2,
    validation_split=0.3
)

In [ ]:
# Get predicted probabilities (or logits) for the test set from the neural network
y_pred = ml.predict(x_te)

In [ ]:
# Print classification report comparing y_te with neural‑network predictions
print(classification_report(y_te, y_pred))

In [ ]:
# Plot training vs validation accuracy and loss over epochs to inspect learning behavior
plt.figure(figsize=(15, 8))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()